# Búsqueda lineal

Recorre la secuencia elemento a elemento hasta encontrar el buscado. Funciona con datos sin ordenar y cuesta $O(n)$.

## 1. Índice de un elemento

Devolver el índice de la primera aparición de un valor o −1 si no está, sin usar `index` ni `in`.

In [1]:
def busqueda_lineal(lista, objetivo):
    for i, valor in enumerate(lista):
        if valor == objetivo:
            return i
    return -1

datos = [7, 3, 9, 3, 1]
print(busqueda_lineal(datos, 3), busqueda_lineal(datos, 5))

1 -1


## 2. Todas las apariciones

Devolver todos los índices donde aparece un valor.

In [2]:
def todas_las_posiciones(lista, objetivo):
    return [i for i, valor in enumerate(lista) if valor == objetivo]

print(todas_las_posiciones([7, 3, 9, 3, 1, 3], 3))

[1, 3, 5]


## 3. Búsqueda con condición

Devolver el primer elemento que cumpla una condición (o `None`), por ejemplo el primer número primo mayor que 100 de una lista.

In [3]:
def primero_que_cumple(lista, condicion):
    return next((x for x in lista if condicion(x)), None)

def es_primo(n):
    return n > 1 and all(n % d for d in range(2, int(n**0.5) + 1))

print(primero_que_cumple([90, 120, 101, 103, 7], lambda x: x > 100 and es_primo(x)))

101


## 4. Búsqueda con centinela

Añadir el objetivo al final de la lista (el centinela) para que el bucle no tenga que comprobar en cada vuelta si se ha llegado al final; quitarlo antes de devolver el resultado.

In [4]:
import random

def busqueda_centinela(lista, objetivo):
    n = len(lista)
    lista.append(objetivo)
    i = 0
    # sin comprobar i < n: el centinela garantiza que el bucle termina
    while lista[i] != objetivo:
        i += 1
    lista.pop()
    return i if i < n else -1

random.seed(4)
for _ in range(500):
    datos = random.choices(range(20), k=random.randint(0, 15))
    copia, x = datos[:], random.randrange(25)
    assert busqueda_centinela(datos, x) == busqueda_lineal(datos, x)
    assert datos == copia
print(busqueda_centinela([7, 3, 9, 3, 1], 9), busqueda_centinela([7, 3, 9, 3, 1], 5))

2 -1


## 5. Apariciones de una sublista

Devolver las posiciones donde empieza una sublista (o subcadena) dentro de otra, incluidas las que se solapan, comparando elemento a elemento sin usar `find` ni rebanadas. Cuesta $O(n \cdot m)$.

In [5]:
def apariciones(secuencia, patron):
    n, m = len(secuencia), len(patron)
    posiciones = []
    for i in range(n - m + 1):
        j = 0
        while j < m and secuencia[i + j] == patron[j]:
            j += 1
        if j == m:
            posiciones.append(i)
    return posiciones

def apariciones_fuerza_bruta(secuencia, patron):
    m = len(patron)
    return [i for i in range(len(secuencia) - m + 1) if secuencia[i:i + m] == patron]

random.seed(5)
for _ in range(500):
    s = random.choices([0, 1], k=random.randint(0, 20))
    p = random.choices([0, 1], k=random.randint(1, 4))
    assert apariciones(s, p) == apariciones_fuerza_bruta(s, p)
print(apariciones([1, 2, 1, 2, 1, 3, 1, 2, 1], [1, 2, 1]))
print(apariciones("abracadabra", "abra"))

[0, 2, 6]
[0, 7]


## 6. Búsqueda en una matriz

Buscar un valor en una matriz (lista de listas) y devolver `(fila, columna)` o `None`. Si cada fila y cada columna están ordenadas de menor a mayor, empezar en la esquina superior derecha: si el valor es mayor que el objetivo se descarta su columna y si es menor, su fila. Así cuesta $O(f + c)$ en vez de $O(f \cdot c)$.

In [6]:
def busca_en_matriz(matriz, objetivo):
    for f, fila in enumerate(matriz):
        for c, valor in enumerate(fila):
            if valor == objetivo:
                return f, c
    return None

def busca_en_escalera(matriz, objetivo):
    f, c, pasos = 0, len(matriz[0]) - 1, 0
    while f < len(matriz) and c >= 0:
        pasos += 1
        if matriz[f][c] == objetivo:
            return (f, c), pasos
        if matriz[f][c] > objetivo:
            c -= 1
        else:
            f += 1
    return None, pasos

def matriz_ordenada(filas, columnas):
    m = [[0] * columnas for _ in range(filas)]
    for f in range(filas):
        for c in range(columnas):
            arriba = m[f - 1][c] if f else 0
            izquierda = m[f][c - 1] if c else 0
            m[f][c] = max(arriba, izquierda) + random.randint(1, 4)
    return m

random.seed(6)
m = matriz_ordenada(4, 5)
for fila in m:
    print(" ".join(f"{v:3}" for v in fila))
print(busca_en_matriz(m, m[2][3]), busca_en_escalera(m, m[2][3]), busca_en_escalera(m, 100))

for _ in range(300):
    m = matriz_ordenada(random.randint(1, 8), random.randint(1, 8))
    x = random.randint(0, 60)
    pos, pasos = busca_en_escalera(m, x)
    assert (pos is None) == (busca_en_matriz(m, x) is None)
    assert pos is None or m[pos[0]][pos[1]] == x
    assert pasos <= len(m) + len(m[0]) - 1

  1   5   8   9  10
  3   9  12  15  16
  6  13  15  19  20
  8  16  17  23  26
(2, 3) ((2, 3), 4) (None, 4)


## 7. Contar comparaciones

Si el objetivo está en la posición $i$, la búsqueda lineal lo compara con $i + 1$ elementos: de media $\frac{n + 1}{2}$ si está y $n$ si no está. Contar también las comparaciones de índice ($i < n$) y comprobar que, cuando el valor no está, la versión normal hace $2n + 1$ comparaciones en total y la del centinela solo $n + 2$.

In [7]:
def lineal_contando(lista, objetivo):
    indices = valores = i = 0
    while True:
        indices += 1
        if i == len(lista):
            return -1, indices, valores
        valores += 1
        if lista[i] == objetivo:
            return i, indices, valores
        i += 1

def centinela_contando(lista, objetivo):
    n, valores, i = len(lista), 0, 0
    lista.append(objetivo)
    while True:
        valores += 1
        if lista[i] == objetivo:
            break
        i += 1
    lista.pop()
    # la única comparación de índice es la final, i < n
    return (i if i < n else -1), 1, valores

print(f"{'n':>5} {'mejor':>6} {'medio':>7} {'ausente':>8} {'ausente con centinela':>22}")
for n in (10, 100, 1000):
    datos = list(range(n))
    presentes = [lineal_contando(datos, x)[2] for x in datos]
    _, indices, valores = lineal_contando(datos, -1)
    _, indices_c, valores_c = centinela_contando(datos, -1)
    assert sum(presentes) / n == (n + 1) / 2
    assert indices + valores == 2 * n + 1 and indices_c + valores_c == n + 2
    print(f"{n:>5} {min(presentes):>6} {sum(presentes) / n:>7} {indices + valores:>8} {indices_c + valores_c:>22}")

    n  mejor   medio  ausente  ausente con centinela
   10      1     5.5       21                     12
  100      1    50.5      201                    102
 1000      1   500.5     2001                   1002
